In [1]:
import pandas as pd
from src.lista4.data import cargar_datos
from src.lista4.models import ajustar_regresion_logistica
from src.lista4.metrics import riesgo_empirico_logistico, errores_clasificacion

Cargar datos

In [2]:
df = cargar_datos('../data/clientes.csv')
y = df['abandona_30d']

Variables disponibles t0

In [3]:
todas_vars = ['antiguedad_meses', 'uso_30d_previos', 'facturas_vencidas_180d', 'tickets_90d', 'pago_automatico', 'ajuste_comercial_30d']
vars_disponibles = ['antiguedad_meses', 'uso_30d_previos', 'facturas_vencidas_180d', 'tickets_90d', 'pago_automatico']

Modelos

In [4]:
mod_M1 = ajustar_regresion_logistica(df[todas_vars], y)
mod_M2 = ajustar_regresion_logistica(df[vars_disponibles], y)

prob_M1 = mod_M1.predict_proba(df[todas_vars])[:, 1]
prob_M2 = mod_M2.predict_proba(df[vars_disponibles])[:, 1]
clase_M1 = (prob_M1 >= 0.5).astype(int)
clase_M2 = (prob_M2 >= 0.5).astype(int)

Riesgos y errores

In [5]:
riesgo_M1 = riesgo_empirico_logistico(y, prob_M1)
riesgo_M2 = riesgo_empirico_logistico(y, prob_M2)

err_num_M1, err_prop_M1 = errores_clasificacion(y, clase_M1)
err_num_M2, err_prop_M2 = errores_clasificacion(y, clase_M2)

Tabla final

In [6]:
tabla_ej4 = pd.DataFrame({
    'Modelo': ['M1 (Todas)', 'M2 (Variables en t0)'],
    'Riesgo Logístico': [riesgo_M1, riesgo_M2],
    'Errores (#)': [err_num_M1, err_num_M2],
    'Proporción Errores': [err_prop_M1, err_prop_M2]
})
display(tabla_ej4)

,Modelo,Riesgo Logístico,Errores (#),Proporción Errores
0,M1 (Todas),0.342485,160,0.133333
1,M2 (Variables en t0),0.571716,336,0.280000


Si el modelo se ejecuta exactamente en $t_0$, únicamente se debe usar el Modelo $M_2$. La variable que produce la diferencia es ajuste_comercial_30d, la cual recolecta datos entre $t_0$ y $t_0 + 30$. Incorporar esta variable causaría data leakage (fuga de datos) porque esa información futura aún es desconocida en la fecha de estimación.